In [1]:
# R1_correspondence.ipynb  (revision stage 1)
# Reconstructed theory (addresses reviewer M1, M2, M3, M6, M8).
#
# Frame: with a WoE bin grid, constrained multivariate componentwise depth-1
# boosting is coordinate descent on a dummy (bin-indicator) additive logistic
# model  eta = b0 + sum_j sum_b gamma_jb 1{x_j in bin b}.  Saturated, it converges
# to the dummy-logistic MLE gamma_hat. Logistic-WoE is its RANK-ONE sub-model
# gamma_jb = beta_j * WoE_j(b). The empirical question (NOT an identity):
#   does the free gamma_hat_j(.) lie close to a scalar multiple of WoE_j(.),
#   and does that multiple (the projection slope) track the WoE-logistic beta_hat_j?
# The univariate (naive-Bayes) limit gives slope == 1 by construction; the
# MULTIVARIATE slope reproduces beta_hat_j, i.e. it captures correlation correction.
#
# All binning + IV selection is done INSIDE each training fold (no leakage, M6).
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED=42; N_SPLIT=20; IV_THRESH=0.02
rng=np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

# ===================================================================
# 0. LOAD GMSC (clean) — raw cleaned values, bin INSIDE each fold
# ===================================================================
dfc=pd.read_parquet(os.path.join(DATA,'gmsc_clean.parquet'))
TARGET='SeriousDlqin2yrs'
FEATURES=[c for c in dfc.columns if c!=TARGET]
y_all=dfc[TARGET].values.astype(int); X_all=dfc[FEATURES]
print('GMSC clean:', dfc.shape, '| bad rate %.4f'%y_all.mean())

# ---- in-fold binning utilities (monotone numeric; missing as own bin) ----
def tree_edges(x,yv,max_leaf=6,min_frac=0.05):
    m=x.notna().values; xx=x[m].values.reshape(-1,1); yy=yv[m]
    if len(np.unique(xx))<=2: return []
    t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
    t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)
def bin_counts(x,yv,edges,TG,TB,min_n):
    cuts=[-np.inf]+list(edges)+[np.inf]; idx=pd.cut(x,cuts,labels=False,include_lowest=True)
    recs=[]
    for b in range(len(cuts)-1):
        sel=idx==b; g=int(((yv==0)&sel).sum()); bd=int(((yv==1)&sel).sum())
        if g+bd>0: recs.append({'lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd,'miss':False})
    selm=x.isna(); gm=int(((yv==0)&selm).sum()); bm=int(((yv==1)&selm).sum())
    if gm+bm>0: recs.append({'lo':np.nan,'hi':np.nan,'good':gm,'bad':bm,'miss':True})
    for r in recs:
        gr=(r['good']+0.5)/(TG+0.5); br=(r['bad']+0.5)/(TB+0.5)
        r['woe']=float(np.log(br/gr)); r['n']=r['good']+r['bad']
        if r['miss'] and r['n']<min_n: r['woe']=0.0
    return recs
def enforce_mono(x,yv,edges,TG,TB,min_n):
    e=list(edges)
    while e:
        recs=[r for r in bin_counts(x,yv,e,TG,TB,min_n) if not r['miss']]
        br=np.array([r['bad']/r['n'] for r in recs])
        if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): return e
        d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
    return e
def fit_binning(Xtr,ytr):
    TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=int(0.01*len(ytr)); maps={}; iv={}
    for c in FEATURES:
        edges=enforce_mono(Xtr[c],ytr,tree_edges(Xtr[c],ytr),TG,TB,min_n)
        recs=bin_counts(Xtr[c],ytr,edges,TG,TB,min_n); maps[c]=recs
        iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in recs]))
    keep=[c for c in FEATURES if iv[c]>=IV_THRESH]
    return maps, keep
def apply_bins(x, recs):
    idx=np.full(len(x),-1,int); fin=[r for r in recs if not r['miss']]
    for p,r in enumerate(fin):
        sel=(~x.isna()).values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
    miss_pos=[p for p,r in enumerate(recs) if r['miss']]
    if miss_pos: idx[x.isna().values]=miss_pos[0]
    else: idx[x.isna().values]=0
    idx[idx==-1]=0
    return idx

# ===================================================================
# 1. REPEATED FOLDS: WoE-logit beta_hat vs dummy-logit gamma_hat slope; LR test
# ===================================================================
def woe_design(Xdf, maps, keep):
    cols=[]; 
    for c in keep:
        recs=maps[c]; woe=np.array([r['woe'] for r in recs]); idx=apply_bins(Xdf[c],recs); cols.append(woe[idx])
    return np.column_stack(cols)
def dummy_design(Xdf, maps, keep):
    cols=[]; key=[]
    for c in keep:
        recs=maps[c]; idx=apply_bins(Xdf[c],recs); K=len(recs)
        for b in range(K-1):                       # drop last bin = reference
            cols.append((idx==b).astype(float)); key.append((c,b,K))
    return np.column_stack(cols), key

skf=StratifiedKFold(n_splits=N_SPLIT, shuffle=True, random_state=SEED)
slope_rows=[]; lr_rows=[]; t0=time.time()
for fold,(tr,te) in enumerate(skf.split(X_all,y_all)):
    Xtr=X_all.iloc[tr]; ytr=y_all[tr]
    maps, keep = fit_binning(Xtr,ytr)
    pop={c:np.array([r['n'] for r in maps[c]],float) for c in keep}
    woeb={c:np.array([r['woe'] for r in maps[c]]) for c in keep}
    # WoE multivariate logistic
    Xw=woe_design(Xtr,maps,keep); mw=sm.Logit(ytr,sm.add_constant(Xw)).fit(disp=0)
    beta={c:mw.params[i+1] for i,c in enumerate(keep)}
    ll_woe=mw.llf; k_woe=len(keep)+1
    # dummy multivariate logistic (near-MLE)
    Xd,key=dummy_design(Xtr,maps,keep)
    clf=LogisticRegression(C=5000,max_iter=4000,solver='lbfgs').fit(Xd,ytr)
    p=np.clip(clf.predict_proba(Xd)[:,1],1e-12,1-1e-12)
    ll_dummy=float(np.sum(ytr*np.log(p)+(1-ytr)*np.log(1-p)))
    k_dummy=Xd.shape[1]+1
    gamma={c:np.zeros(len(maps[c])) for c in keep}
    for (c,b,K),v in zip(key,clf.coef_.ravel()): gamma[c][b]=v
    for c in keep:                                  # center within feature (pop-weighted)
        gamma[c]=gamma[c]-np.sum(pop[c]*gamma[c])/pop[c].sum()
    # per-feature projection slope of gamma on WoE
    for c in keep:
        w=pop[c]; wv=woeb[c]-np.sum(pop[c]*woeb[c])/pop[c].sum()
        den=np.sum(w*wv**2); slope=np.sum(w*gamma[c]*wv)/den if den>0 else np.nan
        slope_rows.append([fold,c,beta[c],slope])
    # LR test: rank-one (WoE) nested in free (dummy)
    LR=2*(ll_dummy-ll_woe); df=k_dummy-k_woe; pval=stats.chi2.sf(max(LR,0),df)
    lr_rows.append([fold,LR,df,pval])
    if (fold+1)%5==0: print('  fold %d/%d (%.0fs)'%(fold+1,N_SPLIT,time.time()-t0))
print('folds done %.0fs'%(time.time()-t0))

slope_df=pd.DataFrame(slope_rows,columns=['fold','feature','beta_hat','proj_slope'])
lr_df=pd.DataFrame(lr_rows,columns=['fold','LR','df','p'])
# aggregate per feature
agg=slope_df.groupby('feature').agg(beta=('beta_hat','mean'),beta_sd=('beta_hat','std'),
       slope=('proj_slope','mean'),slope_sd=('proj_slope','std')).reset_index()
agg['abs_gap']=(agg['beta']-agg['slope']).abs()
print('\n=== Multivariate projection slope vs WoE beta_hat (mean over folds) ===')
print(agg.round(4).to_string(index=False))
rr=np.corrcoef(agg['beta'],agg['slope'])[0,1]
print('\ncorr(beta_hat, proj_slope) = %.4f | mean|gap| = %.4f'%(rr,agg['abs_gap'].mean()))
print('naive-Bayes univariate slope would be 1.00 for every feature (contrast).')
print('\n=== Rank-one (WoE) vs free (dummy) LR test, per fold ===')
print('mean LR=%.1f  df=%d  median p=%.3g'%(lr_df['LR'].mean(),int(lr_df['df'].iloc[0]),lr_df['p'].median()))
print('(small p => WoE rank-one constraint is statistically rejected vs free bins;')
print(' effect SIZE is what matters — see discrimination gap in stage 3.)')

# ===================================================================
# 2. COMPONENTWISE BOOSTING PATH -> dummy-logit MLE (one representative fold)
#    base learner each round: the single feature whose bin-group best reduces
#    log-loss; Newton update on that feature's bin means; shrink by nu.
# ===================================================================
tr,te=next(iter(skf.split(X_all,y_all)))
Xtr=X_all.iloc[tr]; ytr=y_all[tr]; maps,keep=fit_binning(Xtr,ytr)
idxs={c:apply_bins(Xtr[c],maps[c]) for c in keep}; nbk={c:len(maps[c]) for c in keep}
pop={c:np.array([r['n'] for r in maps[c]],float) for c in keep}
# target gamma_hat (dummy MLE) for reference
Xd,key=dummy_design(Xtr,maps,keep); clf=LogisticRegression(C=5000,max_iter=4000).fit(Xd,ytr)
gamma_mle={c:np.zeros(nbk[c]) for c in keep}
for (c,b,K),v in zip(key,clf.coef_.ravel()): gamma_mle[c][b]=v
for c in keep: gamma_mle[c]-=np.sum(pop[c]*gamma_mle[c])/pop[c].sum()
# boosting
nu=0.3; M=400; F0=np.log(ytr.mean()/(1-ytr.mean()))
eta=np.full(len(ytr),F0); gcur={c:np.zeros(nbk[c]) for c in keep}
dist=[]
for m in range(M):
    p=1/(1+np.exp(-eta)); grad=ytr-p; hess=p*(1-p)
    best=None; bestgain=-1; bestdelta=None
    for c in keep:
        num=np.bincount(idxs[c],weights=grad,minlength=nbk[c])
        den=np.bincount(idxs[c],weights=hess,minlength=nbk[c])+1e-9
        delta=num/den; gain=np.sum(num**2/den)
        if gain>bestgain: bestgain=gain; best=c; bestdelta=delta
    gcur[best]=gcur[best]+nu*bestdelta
    eta=eta+nu*bestdelta[idxs[best]]
    if (m+1)%10==0:
        d=0.0
        for c in keep:
            gc=gcur[c]-np.sum(pop[c]*gcur[c])/pop[c].sum()
            d+=np.sum(pop[c]*(gc-gamma_mle[c])**2)
        dist.append([m+1, float(np.sqrt(d/sum(pop[c].sum() for c in keep)))])
dist=np.array(dist)
print('\n=== Componentwise boosting path -> dummy-logit MLE ===')
print('final weighted RMS distance to MLE = %.4f (round %d)'%(dist[-1,1],int(dist[-1,0])))

# ===================================================================
# 3. SAVE + FIGURES
# ===================================================================
agg.round(6).to_csv(os.path.join(TABLES,'R1_slope_vs_beta.csv'),index=False)
lr_df.round(6).to_csv(os.path.join(TABLES,'R1_lr_test.csv'),index=False)
pd.DataFrame(dist,columns=['round','rms_to_mle']).to_csv(os.path.join(TABLES,'R1_boost_path.csv'),index=False)

# Fig A: projection slope vs beta_hat (identity) -- the NEW key figure (Regime II)
fig,ax=plt.subplots(figsize=(5.8,5.6))
lo=min(agg['beta'].min(),agg['slope'].min())*1.1; hi=max(agg['beta'].max(),agg['slope'].max())*1.1
ax.plot([lo,hi],[lo,hi],color='0.0',lw=1,ls='--',label='identity')
ax.axhline(1.0,color='0.6',lw=1,ls=':',label='naive-Bayes slope = 1')
ax.errorbar(agg['beta'],agg['slope'],xerr=agg['beta_sd'],yerr=agg['slope_sd'],
            fmt='o',color='0.25',ecolor='0.6',capsize=2,ms=6)
ax.set_xlabel(r'logistic-WoE coefficient $\hat\beta_j$')
ax.set_ylabel(r'dummy-logit projection slope onto WoE')
ax.set_title('GMSC: multivariate slope tracks $\\hat\\beta_j$ (r=%.3f)'%rr,fontsize=10)
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'R1_slope_vs_beta')

# Fig B: boosting path convergence to MLE
fig,ax=plt.subplots(figsize=(6.4,4.2))
ax.plot(dist[:,0],dist[:,1],color='0.2',marker='o',ms=4,lw=1.3)
ax.set_xlabel('boosting round'); ax.set_ylabel('weighted RMS distance to dummy-logit MLE')
ax.set_title('Componentwise boosting converges to the dummy-logit MLE',fontsize=10)
savefig(fig,'R1_boost_convergence')

print('\nSaved figures: R1_slope_vs_beta, R1_boost_convergence')
print('Saved tables : R1_slope_vs_beta, R1_lr_test, R1_boost_path')
print('\n================ R1 HEADLINE ================')
print('Multivariate projection slope vs beta_hat: corr=%.3f, mean|gap|=%.3f (NOT the naive-Bayes 1.0).'%(rr,agg['abs_gap'].mean()))
print('Componentwise boosting -> dummy-logit MLE (RMS %.3f). Logistic-WoE = rank-one sub-model.'%dist[-1,1])
print('All binning + IV selection done inside each training fold (no leakage).')
print('\nR1 complete.')


GMSC clean: (150000, 11) | bad rate 0.0668


  fold 5/20 (6s)


  fold 10/20 (12s)


  fold 15/20 (17s)


  fold 20/20 (23s)
folds done 23s

=== Multivariate projection slope vs WoE beta_hat (mean over folds) ===
                             feature    beta  beta_sd   slope  slope_sd  abs_gap
                       MonthlyIncome  0.3584   0.0101  0.4350    0.0105   0.0766
                  NumberOfDependents  0.1898   0.0144  0.1819    0.0168   0.0079
     NumberOfOpenCreditLinesAndLoans -0.0687   0.0140 -0.2136    0.0204   0.1449
NumberOfTime30-59DaysPastDueNotWorse  0.6272   0.0021  0.6194    0.0020   0.0078
             NumberOfTimes90DaysLate  0.5996   0.0021  0.6159    0.0022   0.0163
RevolvingUtilizationOfUnsecuredLines  0.6809   0.0020  0.6886    0.0031   0.0077
                                 age  0.4474   0.0049  0.4524    0.0093   0.0050

corr(beta_hat, proj_slope) = 0.9871 | mean|gap| = 0.0380
naive-Bayes univariate slope would be 1.00 for every feature (contrast).

=== Rank-one (WoE) vs free (dummy) LR test, per fold ===
mean LR=256.6  df=21  median p=1.65e-42
(small p => WoE 


=== Componentwise boosting path -> dummy-logit MLE ===
final weighted RMS distance to MLE = 0.0323 (round 400)



Saved figures: R1_slope_vs_beta, R1_boost_convergence
Saved tables : R1_slope_vs_beta, R1_lr_test, R1_boost_path

================ R1 HEADLINE ================
Multivariate projection slope vs beta_hat: corr=0.987, mean|gap|=0.038 (NOT the naive-Bayes 1.0).
Componentwise boosting -> dummy-logit MLE (RMS 0.032). Logistic-WoE = rank-one sub-model.
All binning + IV selection done inside each training fold (no leakage).

R1 complete.
